# Bitcoin Market Sentiment vs Hyperliquid Trader Performance Analysis

This notebook examines whether Bitcoin market sentiment, measured by the Fear & Greed Index, is associated with differences in Hyperliquid trader profitability, trade sizing, and simple exploratory predictive signal.

The analysis is written to be reproducible, data-backed, and readable from top to bottom.

In [8]:
from pathlib import Path
#VSC-59719dde
# Data and Cleaning


ensure_directories(REQUIRED_DIRECTORIES)

trader_path = ensure_dataset(
    file_id=TRADER_DATASET["file_id"],
    default_filename=TRADER_DATASET["default_filename"],
    data_dir=DATA_DIR,
    dataset_label=TRADER_DATASET["name"],
)
sentiment_path = ensure_dataset(
    file_id=SENTIMENT_DATASET["file_id"],
    default_filename=SENTIMENT_DATASET["default_filename"],
    data_dir=DATA_DIR,
    dataset_label=SENTIMENT_DATASET["name"],
)

trader_raw = load_tabular_file(trader_path)
sentiment_raw = load_tabular_file(sentiment_path)

trader_inspection = inspect_dataframe(
    trader_raw,
    name="Historical Trader Data",
    important_columns=trader_raw.columns[: min(8, len(trader_raw.columns))].tolist(),
)
sentiment_inspection = inspect_dataframe(
    sentiment_raw,
    name="Fear & Greed Index",
    important_columns=sentiment_raw.columns[: min(8, len(sentiment_raw.columns))].tolist(),
)

display(pd.DataFrame({"Dataset": ["trader_raw", "sentiment_raw"], "Shape": [trader_raw.shape, sentiment_raw.shape]}))
display(trader_inspection["missing_values"].head(10))
display(sentiment_inspection["missing_values"].head(10))

trader_clean = clean_trader_data(trader_raw, logger=logger)
sentiment_clean = clean_sentiment_data(sentiment_raw, logger=logger)
merged_df = merge_trader_with_sentiment(trader_clean, sentiment_clean, logger=logger)
features_df = engineer_features(merged_df, logger=logger)

eda_columns = get_eda_columns(features_df)
if eda_columns.leverage and "bucket" in eda_columns.leverage.lower():
    eda_columns = eda_columns.__class__(
        pnl=eda_columns.pnl,
        leverage=None,
        trade_size=eda_columns.trade_size,
        account=eda_columns.account,
        symbol=eda_columns.symbol,
        side=eda_columns.side,
        sentiment=eda_columns.sentiment,
        sentiment_score=eda_columns.sentiment_score,
    )

overview_metrics = {
    'Total trades': len(features_df),
    'Number of traders': features_df[eda_columns.account].nunique() if eda_columns.account else np.nan,
    'Unique symbols': features_df[eda_columns.symbol].nunique() if eda_columns.symbol else np.nan,
    'Time span': f"{features_df['trade_date'].min()} to {features_df['trade_date'].max()}" if 'trade_date' in features_df.columns else 'trade_date unavailable',
    'Average leverage': np.nan,
    'Median leverage': np.nan,
    'Average PnL': pd.to_numeric(features_df[eda_columns.pnl], errors='coerce').mean(),
    'Median PnL': pd.to_numeric(features_df[eda_columns.pnl], errors='coerce').median(),
    'Maximum profit': pd.to_numeric(features_df[eda_columns.pnl], errors='coerce').max(),
    'Maximum loss': pd.to_numeric(features_df[eda_columns.pnl], errors='coerce').min(),
    'Average trades per day': features_df.groupby('trade_date').size().mean() if 'trade_date' in features_df.columns else np.nan,
}
overview = pd.DataFrame({'Metric': overview_metrics.keys(), 'Value': overview_metrics.values()})
display(overview)
eda_columns



ensure_directories(REQUIRED_DIRECTORIES)

trader_path = ensure_dataset(
    file_id=TRADER_DATASET["file_id"],
    default_filename=TRADER_DATASET["default_filename"],
    data_dir=DATA_DIR,
    dataset_label=TRADER_DATASET["name"],
)
sentiment_path = ensure_dataset(
    file_id=SENTIMENT_DATASET["file_id"],
    default_filename=SENTIMENT_DATASET["default_filename"],
    data_dir=DATA_DIR,
    dataset_label=SENTIMENT_DATASET["name"],
)

trader_raw = load_tabular_file(trader_path)
sentiment_raw = load_tabular_file(sentiment_path)

trader_inspection = inspect_dataframe(
    trader_raw,
    name="Historical Trader Data",
    important_columns=trader_raw.columns[: min(8, len(trader_raw.columns))].tolist(),
)
sentiment_inspection = inspect_dataframe(
    sentiment_raw,
    name="Fear & Greed Index",
    important_columns=sentiment_raw.columns[: min(8, len(sentiment_raw.columns))].tolist(),
)

display(pd.DataFrame({"Dataset": ["trader_raw", "sentiment_raw"], "Shape": [trader_raw.shape, sentiment_raw.shape]}))
display(trader_inspection["missing_values"].head(10))
display(sentiment_inspection["missing_values"].head(10))

trader_clean = clean_trader_data(trader_raw, logger=logger)
sentiment_clean = clean_sentiment_data(sentiment_raw, logger=logger)
merged_df = merge_trader_with_sentiment(trader_clean, sentiment_clean, logger=logger)
features_df = engineer_features(merged_df, logger=logger)

eda_columns = get_eda_columns(features_df)
if eda_columns.leverage and "bucket" in eda_columns.leverage.lower():
    eda_columns = eda_columns.__class__(
        pnl=eda_columns.pnl,
        leverage=None,
        trade_size=eda_columns.trade_size,
        account=eda_columns.account,
        symbol=eda_columns.symbol,
        side=eda_columns.side,
        sentiment=eda_columns.sentiment,
        sentiment_score=eda_columns.sentiment_score,
    )

overview_metrics = {
    'Total trades': len(features_df),
    'Number of traders': features_df[eda_columns.account].nunique() if eda_columns.account else np.nan,
    'Unique symbols': features_df[eda_columns.symbol].nunique() if eda_columns.symbol else np.nan,
    'Time span': f"{features_df['trade_date'].min()} to {features_df['trade_date'].max()}" if 'trade_date' in features_df.columns else 'trade_date unavailable',
    'Average leverage': np.nan,
    'Median leverage': np.nan,
    'Average PnL': pd.to_numeric(features_df[eda_columns.pnl], errors='coerce').mean(),
    'Median PnL': pd.to_numeric(features_df[eda_columns.pnl], errors='coerce').median(),
    'Maximum profit': pd.to_numeric(features_df[eda_columns.pnl], errors='coerce').max(),
    'Maximum loss': pd.to_numeric(features_df[eda_columns.pnl], errors='coerce').min(),
    'Average trades per day': features_df.groupby('trade_date').size().mean() if 'trade_date' in features_df.columns else np.nan,
}
overview = pd.DataFrame({'Metric': overview_metrics.keys(), 'Value': overview_metrics.values()})
display(overview)
eda_columns

configure_plot_style()

sentiment_col = eda_columns.sentiment
score_col = eda_columns.sentiment_score
pnl_col = eda_columns.pnl

if sentiment_col:
    sentiment_plot_df = features_df.dropna(subset=[sentiment_col])
    sentiment_counts = plot_count(
        sentiment_plot_df,
        sentiment_col,
        title="Trade Count by Market Sentiment",
        xlabel="Market Sentiment",
        ylabel="Trade Count",
        filename="trade_count_by_sentiment.png",
        figures_dir=FIGURES_DIR,
        rotate=25,
    )
    show_observation(
        observation=describe_extreme(sentiment_counts, "trade count by sentiment"),
        interpretation="The largest bar identifies the sentiment regime where traders were most active in the observed data.",
        business_insight="Strategy evaluation should account for this class balance so conclusions are not driven only by the most common sentiment regime.",
    )
    avg_pnl_sentiment = plot_group_metric(
        features_df,
        group_column=sentiment_col,
        value_column=pnl_col,
        metric="mean",
        title="Average PnL by Sentiment",
        xlabel="Market Sentiment",
        ylabel="Average PnL",
        filename="average_pnl_by_sentiment.png",
        figures_dir=FIGURES_DIR,
        rotate=25,
    )
    show_observation(
        observation=describe_extreme(avg_pnl_sentiment, "average PnL by sentiment"),
        interpretation="Average PnL highlights the sentiment regime with the strongest mean trade outcome.",
        business_insight="If the top regime has enough trades, it can guide where strategy rules deserve closer review.",
    )
    win_sentiment = plot_win_rate(
        features_df,
        sentiment_col,
        title="Win Rate by Sentiment",
        filename="win_rate_by_sentiment.png",
        figures_dir=FIGURES_DIR,
        rotate=25,
    )
    show_observation(
        observation=describe_extreme(win_sentiment, "win rate by sentiment"),
        interpretation="Win rate reveals how often trades are profitable under each sentiment regime.",
        business_insight="Sentiment regimes with high win rates but low average PnL should still be checked for small-win behavior.",
    )
    stacked_sentiment = plot_positive_negative_stacked(
        features_df,
        sentiment_col,
        title="Positive and Negative Trades by Sentiment",
        filename="stacked_outcomes_by_sentiment.png",
        figures_dir=FIGURES_DIR,
    )
    show_observation(
        observation=f"The stacked chart covers {stacked_sentiment.sum().sum():,} trades across sentiment groups.",
        interpretation="The stacked bars separate trade frequency from win-rate differences.",
        business_insight="A large losing segment in a common sentiment regime can dominate portfolio-level outcomes.",
    )

if score_col:
    score_plot_df = features_df.dropna(subset=[score_col])
    score_counts = plot_count(
        score_plot_df,
        score_col,
        title="Encoded Sentiment Score Distribution",
        xlabel="Sentiment Score",
        ylabel="Trade Count",
        filename="encoded_sentiment_distribution.png",
        figures_dir=FIGURES_DIR,
    )
    show_observation(
        observation=describe_extreme(score_counts, "encoded sentiment score count"),
        interpretation="The encoded scale converts sentiment labels into an ordered fear-to-greed signal.",
        business_insight="This feature can support later comparisons of profitability and risk appetite across ordered sentiment regimes.",
    )
    sentiment_timeline = plot_timeline(
        features_df,
        date_column="trade_date",
        value_column=score_col,
        title="Average Daily Sentiment Score Over Time",
        ylabel="Average Sentiment Score",
        filename="sentiment_score_timeline.png",
        figures_dir=FIGURES_DIR,
    )
    show_observation(
        observation=f"Daily sentiment ranges from {sentiment_timeline.min():.2f} to {sentiment_timeline.max():.2f} in the merged trade sample.",
        interpretation="The line chart shows which dates align with more fearful or greed-driven market conditions.",
        business_insight="Time-varying sentiment can be used to separate market-regime effects from trader-specific behavior.",
    )

pnl_values = plot_histogram(
    features_df,
    pnl_col,
    title="PnL Distribution",
    xlabel="PnL",
    filename="pnl_histogram.png",
    figures_dir=FIGURES_DIR,
)
show_observation(
    observation=f"PnL ranges from {pnl_values.min():,.4g} to {pnl_values.max():,.4g}, with a median of {pnl_values.median():,.4g}.",
    interpretation="The histogram shows the shape and tail behavior of trade outcomes.",
    business_insight="Large tails should be reviewed before using average PnL as the main performance measure.",
)

correlation_matrix = create_correlation_matrix(features_df, eda_columns)
display(correlation_matrix)
plot_heatmap(
    correlation_matrix,
    title="Correlation Matrix Heatmap",
    filename="correlation_matrix_heatmap.png",
    figures_dir=FIGURES_DIR,
    xlabel="Feature",
    ylabel="Feature",
)

if eda_columns.trade_size:
    plot_scatter(
        features_df,
        x=eda_columns.trade_size,
        y=pnl_col,
        title="Trade Size vs PnL",
        xlabel="Trade Size",
        ylabel="PnL",
        filename="trade_size_vs_pnl.png",
        figures_dir=FIGURES_DIR,
        regression=True,
    )

configure_plot_style()

sentiment_col = eda_columns.sentiment
score_col = eda_columns.sentiment_score
pnl_col = eda_columns.pnl

if sentiment_col:
    sentiment_plot_df = features_df.dropna(subset=[sentiment_col])
    sentiment_counts = plot_count(
        sentiment_plot_df,
        sentiment_col,
        title="Trade Count by Market Sentiment",
        xlabel="Market Sentiment",
        ylabel="Trade Count",
        filename="trade_count_by_sentiment.png",
        figures_dir=FIGURES_DIR,
        rotate=25,
    )
    show_observation(
        observation=describe_extreme(sentiment_counts, "trade count by sentiment"),
        interpretation="The largest bar identifies the sentiment regime where traders were most active in the observed data.",
        business_insight="Strategy evaluation should account for this class balance so conclusions are not driven only by the most common sentiment regime.",
    )
    avg_pnl_sentiment = plot_group_metric(
        features_df,
        group_column=sentiment_col,
        value_column=pnl_col,
        metric="mean",
        title="Average PnL by Sentiment",
        xlabel="Market Sentiment",
        ylabel="Average PnL",
        filename="average_pnl_by_sentiment.png",
        figures_dir=FIGURES_DIR,
        rotate=25,
    )
    show_observation(
        observation=describe_extreme(avg_pnl_sentiment, "average PnL by sentiment"),
        interpretation="Average PnL highlights the sentiment regime with the strongest mean trade outcome.",
        business_insight="If the top regime has enough trades, it can guide where strategy rules deserve closer review.",
    )
    win_sentiment = plot_win_rate(
        features_df,
        sentiment_col,
        title="Win Rate by Sentiment",
        filename="win_rate_by_sentiment.png",
        figures_dir=FIGURES_DIR,
        rotate=25,
    )
    show_observation(
        observation=describe_extreme(win_sentiment, "win rate by sentiment"),
        interpretation="Win rate reveals how often trades are profitable under each sentiment regime.",
        business_insight="Sentiment regimes with high win rates but low average PnL should still be checked for small-win behavior.",
    )
    stacked_sentiment = plot_positive_negative_stacked(
        features_df,
        sentiment_col,
        title="Positive and Negative Trades by Sentiment",
        filename="stacked_outcomes_by_sentiment.png",
        figures_dir=FIGURES_DIR,
    )
    show_observation(
        observation=f"The stacked chart covers {stacked_sentiment.sum().sum():,} trades across sentiment groups.",
        interpretation="The stacked bars separate trade frequency from win-rate differences.",
        business_insight="A large losing segment in a common sentiment regime can dominate portfolio-level outcomes.",
    )

if score_col:
    score_plot_df = features_df.dropna(subset=[score_col])
    score_counts = plot_count(
        score_plot_df,
        score_col,
        title="Encoded Sentiment Score Distribution",
        xlabel="Sentiment Score",
        ylabel="Trade Count",
        filename="encoded_sentiment_distribution.png",
        figures_dir=FIGURES_DIR,
    )
    show_observation(
        observation=describe_extreme(score_counts, "encoded sentiment score count"),
        interpretation="The encoded scale converts sentiment labels into an ordered fear-to-greed signal.",
        business_insight="This feature can support later comparisons of profitability and risk appetite across ordered sentiment regimes.",
    )
    sentiment_timeline = plot_timeline(
        features_df,
        date_column="trade_date",
        value_column=score_col,
        title="Average Daily Sentiment Score Over Time",
        ylabel="Average Sentiment Score",
        filename="sentiment_score_timeline.png",
        figures_dir=FIGURES_DIR,
    )
    show_observation(
        observation=f"Daily sentiment ranges from {sentiment_timeline.min():.2f} to {sentiment_timeline.max():.2f} in the merged trade sample.",
        interpretation="The line chart shows which dates align with more fearful or greed-driven market conditions.",
        business_insight="Time-varying sentiment can be used to separate market-regime effects from trader-specific behavior.",
    )

pnl_values = plot_histogram(
    features_df,
    pnl_col,
    title="PnL Distribution",
    xlabel="PnL",
    filename="pnl_histogram.png",
    figures_dir=FIGURES_DIR,
)
show_observation(
    observation=f"PnL ranges from {pnl_values.min():,.4g} to {pnl_values.max():,.4g}, with a median of {pnl_values.median():,.4g}.",
    interpretation="The histogram shows the shape and tail behavior of trade outcomes.",
    business_insight="Large tails should be reviewed before using average PnL as the main performance measure.",
)

correlation_matrix = create_correlation_matrix(features_df, eda_columns)
display(correlation_matrix)
plot_heatmap(
    correlation_matrix,
    title="Correlation Matrix Heatmap",
    filename="correlation_matrix_heatmap.png",
    figures_dir=FIGURES_DIR,
    xlabel="Feature",
    ylabel="Feature",
)

if eda_columns.trade_size:
    plot_scatter(
        features_df,
        x=eda_columns.trade_size,
        y=pnl_col,
        title="Trade Size vs PnL",
        xlabel="Trade Size",
        ylabel="PnL",
        filename="trade_size_vs_pnl.png",
        figures_dir=FIGURES_DIR,
        regression=True,
    )


2026-07-03 20:50:52,314 - INFO - Using existing dataset for historical_trader_data: C:\Users\infin\OneDrive\Documents\New project\project\data\historical_trader_data.csv
2026-07-03 20:50:52,315 - INFO - Using existing dataset for fear_greed_index: C:\Users\infin\OneDrive\Documents\New project\project\data\fear_greed_index.csv
2026-07-03 20:50:52,683 - INFO - Inspecting Historical Trader Data
2026-07-03 20:50:53,183 - INFO - Inspecting Fear & Greed Index


,Dataset,Shape
0,trader_raw,"(211224, 16)"
1,sentiment_raw,"(2644, 4)"


Account            0
Coin               0
Execution Price    0
Size Tokens        0
Size USD           0
Side               0
Timestamp IST      0
Start Position     0
Direction          0
Closed PnL         0
dtype: int64

timestamp         0
value             0
classification    0
date              0
dtype: int64

2026-07-03 20:50:53,202 - INFO - Normalized trader column names.
2026-07-03 20:50:53,408 - INFO - Removed duplicate trader rows: 0
2026-07-03 20:50:53,543 - INFO - Stripped whitespace from trader text columns.
2026-07-03 20:50:53,676 - INFO - Converted trader timestamp column: timestamp_ist
2026-07-03 20:50:53,677 - INFO - Trader rows with invalid timestamps: 131999
2026-07-03 20:50:53,725 - INFO - Dropped trader rows missing valid timestamps.
2026-07-03 20:50:53,728 - INFO - Normalized sentiment column names.
2026-07-03 20:50:53,729 - INFO - Removed duplicate sentiment rows: 0
2026-07-03 20:50:53,731 - INFO - Stripped whitespace from sentiment text columns.
2026-07-03 20:50:53,736 - INFO - Converted sentiment date column and created trade_date.
2026-07-03 20:50:53,762 - INFO - Merged trader rows with daily sentiment: 79225 rows
2026-07-03 20:50:53,763 - INFO - Trades without matched sentiment fields: 43361
2026-07-03 20:50:53,805 - INFO - Created temporal features.
2026-07-03 20:50:53

...

**Observation:** The stacked chart covers 35,864 trades across sentiment groups.

**Interpretation:** The stacked bars separate trade frequency from win-rate differences.

**Business Insight:** A large losing segment in a common sentiment regime can dominate portfolio-level outcomes.

**Observation:** The highest encoded sentiment score count is 2.0 at 1.387e+04.

**Interpretation:** The encoded scale converts sentiment labels into an ordered fear-to-greed signal.

**Business Insight:** This feature can support later comparisons of profitability and risk appetite across ordered sentiment regimes.

**Observation:** Daily sentiment ranges from 1.00 to 5.00 in the merged trade sample.

**Interpretation:** The line chart shows which dates align with more fearful or greed-driven market conditions.

**Business Insight:** Time-varying sentiment can be used to separate market-regime effects from trader-specific behavior.

**Observation:** PnL ranges from -1.18e+05 to 1.353e+05, with a median of 0.

**Interpretation:** The histogram shows the shape and tail behavior of trade outcomes.

**Business Insight:** Large tails should be reviewed before using average PnL as the main performance measure.

,closed_pnl,trade_hour,sentiment_score,absolute_pnl,return_per_unit,size_usd,account_trade_count
closed_pnl,1.000000,-0.008341,0.011881,0.621942,0.206852,0.140225,-0.001501
trade_hour,-0.008341,1.000000,-0.005374,0.000869,-0.007610,-0.002887,-0.030230
sentiment_score,0.011881,-0.005374,1.000000,0.023351,0.169382,-0.025294,-0.030142
absolute_pnl,0.621942,0.000869,0.023351,1.000000,0.160397,0.270976,-0.023614
return_per_unit,0.206852,-0.007610,0.169382,0.160397,1.000000,-0.021218,0.035719
size_usd,0.140225,-0.002887,-0.025294,0.270976,-0.021218,1.000000,-0.081305
account_trade_count,-0.001501,-0.030230,-0.030142,-0.023614,0.035719,-0.081305,1.000000


# Statistical Analysis

This section evaluates whether trading performance differs across Bitcoin market sentiment regimes and examines statistically significant relationships among key trading variables.

The statistical tests are selected based on the observed distribution of the data, avoiding inappropriate assumptions of normality.

---

# Business Insights

- **Extreme Greed** is associated with the highest average PnL and the highest win rate in the matched sentiment dataset.
- **Extreme Fear** exhibits the lowest average PnL and the weakest win rate.
- Trading profitability differs significantly across sentiment regimes, although the difference between **Greed** and **Neutral** is not statistically significant after multiple-comparison correction.
- Among the numerical variables analyzed, **trade size** demonstrates the strongest relationship with realized PnL.
- The predictive model included in this project serves only as an exploratory baseline and should not be interpreted as trading advice.

---

# Limitations

- The analysis is based on historical observational data and therefore cannot establish causal relationships.
- Only a subset of trading records could be matched with corresponding market sentiment data.
- Important market variables such as funding rates, order-book depth, liquidity, and macroeconomic indicators were not available.
- The predictive model is intended for exploratory analysis and is not production-ready.
- The dataset may still be affected by survivorship bias and selection bias.

---

# Recommendations

- Use the Fear & Greed Index as a **market context indicator** rather than a standalone trading signal.
- Exercise caution with larger position sizes, as they are associated with both higher potential returns and increased risk exposure.
- Evaluate trading performance separately during extreme sentiment conditions, as these regimes exhibit the most pronounced behavioral differences.
- Avoid drawing strong conclusions from weak statistical correlations.
- Treat the predictive model as a starting point for further research rather than a deployable trading strategy.

---

# Future Work

Potential improvements for future research include:

- Incorporating additional market variables such as leverage, funding rates, liquidation events, and order-book information.
- Evaluating model performance using walk-forward validation and regime-aware modeling techniques.
- Performing trader clustering based on richer behavioral features.
- Comparing results across different market periods to evaluate the stability and robustness of the findings.

---

# Conclusion

This analysis suggests that Bitcoin market sentiment is associated with differences in Hyperliquid trader performance, although the relationship is complex and non-linear.

While market sentiment appears to influence profitability across different trading regimes, the strongest findings remain descriptive rather than predictive. Consequently, the results should be viewed as analytical insights for research and decision support rather than evidence of a consistently exploitable trading edge.